# 01 — Colab Pipeline Runner

Run this after preprocessing has already produced `processedImages/` and `img_labels.csv` in Google Drive.

This notebook mounts Drive, clones or updates the repo, installs dependencies, aligns images, and builds the HuggingFace datasets for LoRA and ControlNet training.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

## 2. Clone Or Update Repo

In [ ]:
REPO_URL = 'https://github.com/gabeweng/image-style-transfer.git'
REPO_DIR = '/content/image-style-transfer'

import os

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

## 3. Install Dependencies

In [2]:
!pip install -q uv
!uv pip install --system pillow pillow-heif pandas opencv-python-headless scikit-image

Using Python 3.13.12 environment at: /home/taoren/miniforge3
Resolved 14 packages in 195ms                                        
⠙ Preparing packages... (0/9)                                                   
⠙ Preparing packages... (0/9)-------------------     0 B/7.86 KiB            
⠙ Preparing packages... (0/9)---------- 7.86 KiB/7.86 KiB           
⠙ Preparing packages... (0/9)---------- 7.86 KiB/7.86 KiB           
lazy-loader          ------------------------------ 7.86 KiB/7.86 KiB
⠙ Preparing packages... (0/9)-------------------     0 B/6.75 MiB            
lazy-loader          ------------------------------ 7.86 KiB/7.86 KiB
⠙ Preparing packages... (0/9)-------------------     0 B/6.75 MiB            
lazy-loader          ------------------------------ 7.86 KiB/7.86 KiB
⠙ Preparing packages... (0/9)-------------------     0 B/6.75 MiB            
lazy-loader          ------------------------------ 7.86 KiB/7.86 KiB
imageio              ------------------------------     0 

## 4. Configure Paths

In [ ]:
BASE = '/content/drive/My Drive/CIS_5190_group_project'

PROCESSED_DIR = f'{BASE}/processedImages'
LABELS_CSV = f'{BASE}/img_labels.csv'

ALIGNED_DIR = f'{BASE}/aligned'
ALIGNED_CSV = f'{BASE}/aligned_labels.csv'

HF_DATASET_DIR = f'{BASE}/data/hf_dataset'
HF_CONTROLNET_DIR = f'{BASE}/data/hf_dataset_controlnet'

print('Processed images:', PROCESSED_DIR)
print('Labels CSV:', LABELS_CSV)
print('Aligned images:', ALIGNED_DIR)
print('Aligned CSV:', ALIGNED_CSV)
print('HF dataset:', HF_DATASET_DIR)
print('HF ControlNet dataset:', HF_CONTROLNET_DIR)

## 5. Sanity Check Preprocessing Outputs

In [ ]:
import os
import pandas as pd

assert os.path.isdir(PROCESSED_DIR), f'Missing processed image directory: {PROCESSED_DIR}'
assert os.path.exists(LABELS_CSV), f'Missing labels CSV: {LABELS_CSV}'

labels_df = pd.read_csv(LABELS_CSV)
print(f'Label rows: {len(labels_df)}')
print('Columns:', list(labels_df.columns))
display(labels_df.head())

required_cols = {'file_name', 'location', 'time_of_day', 'weather'}
missing_cols = required_cols - set(labels_df.columns)
assert not missing_cols, f'Missing required columns: {missing_cols}'

missing_files = [
    fname for fname in labels_df['file_name'].head(20)
    if not os.path.exists(os.path.join(PROCESSED_DIR, fname))
]
assert not missing_files, f'Some processed files listed in img_labels.csv were not found: {missing_files[:5]}'

print('Preprocessing outputs look usable.')

## 6. Align Images

This writes homography-aligned images and `aligned_labels.csv` to Drive. Expect console output per location, including `[OK]`, `[FAIL]`, or `[SKIP]` lines.

In [ ]:
!python scripts/align_images.py \
  --images_dir "$PROCESSED_DIR" \
  --labels_csv "$LABELS_CSV" \
  --output_dir "$ALIGNED_DIR" \
  --output_csv "$ALIGNED_CSV" \
  --size 512

## 7. Inspect Alignment Output

In [ ]:
assert os.path.exists(ALIGNED_CSV), f'Missing aligned CSV: {ALIGNED_CSV}'
aligned_df = pd.read_csv(ALIGNED_CSV)
print(f'Aligned pairs: {len(aligned_df)}')
display(aligned_df.head())

if len(aligned_df):
    print(aligned_df.groupby(['target_tod', 'target_weather']).size().unstack(fill_value=0))

## 8. Build Standard HuggingFace Dataset

This creates the dataset used for LoRA or standard Stable Diffusion image training.

In [ ]:
!python scripts/prepare_hf_dataset.py \
  --aligned_csv "$ALIGNED_CSV" \
  --aligned_dir "$ALIGNED_DIR" \
  --output_dir "$HF_DATASET_DIR"

## 9. Build ControlNet HuggingFace Dataset

This creates the same image dataset plus Canny edge maps under `conditioning_images/`.

In [ ]:
!python scripts/prepare_hf_dataset.py \
  --aligned_csv "$ALIGNED_CSV" \
  --aligned_dir "$ALIGNED_DIR" \
  --output_dir "$HF_CONTROLNET_DIR" \
  --controlnet

## 10. Final Dataset Checks

In [ ]:
for path in [
    f'{HF_DATASET_DIR}/metadata.jsonl',
    f'{HF_CONTROLNET_DIR}/metadata.jsonl',
]:
    assert os.path.exists(path), f'Missing metadata file: {path}'
    with open(path) as f:
        n = sum(1 for _ in f)
    print(f'{path}: {n} rows')

print('Pipeline artifacts are ready in Google Drive.')

## 11. GPU Check

Run the remaining training and inference sections on a GPU runtime.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 12. Install Training Dependencies

This installs the heavier diffusion/training stack and pins Hugging Face packages to versions compatible with this repo.

In [ ]:
!uv pip install --system torch torchvision diffusers==0.27.2 'transformers>=4.38.0,<5' 'huggingface-hub<0.26' accelerate peft datasets safetensors lpips streamlit tqdm wandb tensorboard nbconvert ipykernel

## 12.1 Optional W&B Tracking

Set `USE_WANDB = True` if you want diffusion training metrics and checkpoints logged to Weights & Biases. You will be prompted to paste your W&B API key.

In [ ]:
USE_WANDB = False
WANDB_PROJECT = 'image-style-transfer'

if USE_WANDB:
    import os
    os.environ['WANDB_PROJECT'] = WANDB_PROJECT
    os.environ['WANDB_LOG_MODEL'] = 'checkpoint'
    !wandb login
    REPORT_TO_ARG = '--report_to=wandb'
else:
    REPORT_TO_ARG = ''

print('W&B enabled:', USE_WANDB)

## 13. Train Condition Classifier

This trains `checkpoints/classifier_best.pt`, which is later used by evaluation to compute target condition accuracy.

In [ ]:
!python scripts/train_classifier.py \
  --images_dir "$ALIGNED_DIR" \
  --labels_csv "$ALIGNED_CSV" \
  --output_dir "$BASE/checkpoints" \
  --epochs 20 \
  --batch_size 32 \
  --resume

## 14. Install Diffusers Training Examples

The LoRA and ControlNet training scripts live in the Hugging Face `diffusers` examples directory.

In [ ]:
DIFFUSERS_DIR = '/content/diffusers'

if os.path.exists(DIFFUSERS_DIR):
    %cd {DIFFUSERS_DIR}
    !git fetch --tags
    !git checkout v0.27.2
else:
    %cd /content
    !git clone --branch v0.27.2 --depth 1 https://github.com/huggingface/diffusers.git {DIFFUSERS_DIR}

%cd {REPO_DIR}

## 15. Configure Accelerate

This writes a basic single-GPU config so `accelerate launch` can run without the interactive setup prompt.

In [ ]:
!accelerate config default

## 16. Train Stable Diffusion LoRA

This writes LoRA weights to `checkpoints/lora`. It saves frequent checkpoints and resumes from the latest checkpoint if Colab disconnects. The Hugging Face training script includes tqdm progress bars by default.

In [ ]:
LORA_DIR = f'{BASE}/checkpoints/lora'

!accelerate launch /content/diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="$HF_DATASET_DIR" \
  --output_dir="$LORA_DIR" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --num_train_epochs=10 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --mixed_precision="fp16" \
  --gradient_checkpointing \
  --checkpointing_steps=100 \
  --checkpoints_total_limit=3 \
  --resume_from_checkpoint="latest" \
  --caption_column="text" \
  $REPORT_TO_ARG

## 17. Train ControlNet

This writes a fine-tuned ControlNet checkpoint to `checkpoints/controlnet`. It saves frequent checkpoints and resumes from the latest checkpoint if Colab disconnects. This is usually more expensive than LoRA training.

In [ ]:
CONTROLNET_DIR = f'{BASE}/checkpoints/controlnet'

!accelerate launch /content/diffusers/examples/controlnet/train_controlnet.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --output_dir="$CONTROLNET_DIR" \
  --train_data_dir="$HF_CONTROLNET_DIR" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=2 \
  --num_train_epochs=5 \
  --mixed_precision="fp16" \
  --gradient_checkpointing \
  --checkpointing_steps=100 \
  --checkpoints_total_limit=3 \
  --resume_from_checkpoint="latest" \
  --conditioning_image_column="conditioning_images" \
  --image_column="images" \
  --caption_column="text" \
  $REPORT_TO_ARG

## 18. Create Or Load Evaluation Set

The ideal evaluation set is manually curated with `scripts/audit_app.py`. If you have not curated one yet, this creates a practical starter eval set from the best-SSIM aligned pairs.

In [ ]:
EVAL_CSV = f'{BASE}/lpips_eval_set.csv'

if os.path.exists(EVAL_CSV):
    eval_df = pd.read_csv(EVAL_CSV)
    print(f'Using existing eval set: {EVAL_CSV} ({len(eval_df)} rows)')
else:
    eval_df = aligned_df.sort_values('ssim_score', ascending=False).head(min(50, len(aligned_df)))
    eval_df.to_csv(EVAL_CSV, index=False)
    print(f'Created starter eval set: {EVAL_CSV} ({len(eval_df)} rows)')

display(eval_df.head())

## 19. Run Inference Notebook

This executes `02_inference.ipynb` end to end and writes generated images under `outputs/`. Edit the flags inside `02_inference.ipynb` if you want to skip expensive models.

In [ ]:
!jupyter nbconvert --to notebook --execute notebooks/02_inference.ipynb \
  --output 02_inference_executed.ipynb \
  --ExecutePreprocessor.timeout=-1

## 20. Run Evaluation Notebook

This computes LPIPS and condition accuracy using generated outputs and the classifier checkpoint.

In [ ]:
!jupyter nbconvert --to notebook --execute notebooks/03_evaluate.ipynb \
  --output 03_evaluate_executed.ipynb \
  --ExecutePreprocessor.timeout=-1

## 21. Final Artifacts

These are the main files and folders to inspect after the full Colab run finishes.

In [ ]:
expected_outputs = [
    ALIGNED_CSV,
    f'{HF_DATASET_DIR}/metadata.jsonl',
    f'{HF_CONTROLNET_DIR}/metadata.jsonl',
    f'{BASE}/checkpoints/classifier_best.pt',
    f'{BASE}/checkpoints/classifier_last.pt',
    LORA_DIR,
    CONTROLNET_DIR,
    f'{BASE}/outputs',
    f'{BASE}/outputs/evaluation_summary.png',
]

for path in expected_outputs:
    print(('OK     ' if os.path.exists(path) else 'MISSING'), path)